# CLIP

## 参考

CLIP:

- [Learning Transferable Visual Models From Natural Language Supervision][1]

Vision Transformer:

- [AN IMAGE IS WORTH 16x16 WORDS: TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE][2]

ResNet:

- [Deep Residual Learning for Image Recognition][3]
- [Bag of Tricks for Image Classification with Convolutional Neural Networks][4]

[1]: https://arxiv.org/abs/2103.00020
[2]: https://arxiv.org/pdf/2010.11929
[3]: https://arxiv.org/abs/1512.03385
[4]: https://arxiv.org/pdf/1812.01187

## 概要

## 実装

### アーキテクチャ

CLIPの画像エンコーダーは、ResNetとVision Transformerの2種類がある

#### オリジナルのResNet

ResNetは、ボトルネック構造と残差接続を特徴とした画像エンコーダー

ボトルネック構造（右）は、計算量を抑えつつ高い表現力を維持できる3層の畳込み層

入力チャネル数が256の場合:

1. 1x1の畳み込み層でチャンネル数64に圧縮し、ReLU活性化を適用
2. 3x3の畳み込み層で特徴抽出し、ReLU活性化を適用
3. 1x1の畳み込み層でチャンネル数を256に戻し、ReLU活性化関数を適用
4. 残差接続を適用

![](image/bottoleneck.png)

ResNet-34の場合、33層の畳み込み層と全結合層で構成され、残差接続により勾配消失を防いでいた:

![](image/resnet.png)

CLIPの実装では、改良版のResNet-Dにも基づいている:

![](image/resnet1.png)

![](image/resnet2.png)

#### Vision Transformer

Vision Transformerは、単語の代わりに画像のパッチをした言語モデル:

1. 入力の画像をパッチに分割
2. パッチをトークンとみなし、シーケンスをTransformerの次元に射影
3. 学習可能なクラストークンを先頭に追加し、位置埋め込みを加算（CLIPの場合は、EoTトークン）
4. L個のTransformerを適用し、クラストークンが存在する位置のベクトルを抽出
5. MLPを適用し、クラスごとのロジットを得る

![](image/vision_transformer.png)

### 環境構築

In [ ]:
%pip install -q ftfy regex tqdm scikit-image

from collections import OrderedDict
from collections import OrderedDict
from functools import lru_cache
from packaging import version
from PIL import Image
from torch import nn
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize
from tqdm import tqdm
from typing import Tuple, Union, Union, List
import ftfy
import gzip
import hashlib
import html
import IPython.display
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
import regex as re
import skimage
import torch
import torch.nn.functional as F
import urllib
import warnings

try:
    from torchvision.transforms import InterpolationMode
    BICUBIC = InterpolationMode.BICUBIC
except ImportError:
    BICUBIC = Image.BICUBIC

if not os.path.exists("clip"):
    !git clone https://github.com/openai/CLIP.git

%load_ext autoreload
%autoreload 2

%matplotlib inline
%config InlineBackend.figure_format = "retina"
matplotlib.rcParams["font.size"] = 10

In [ ]:
# ログ設定

import logging as logging_

if os.path.exists("debug.log"):
    os.remove("debug.log")

def custom_format(record):
    match record.levelno:
        case logging_.DEBUG:
            level = "🟦"
        case logging_.INFO:
            level = "🟩"
        case logging_.WARNING:
            level = "🟨"
        case logging_.ERROR:
            level = "🟥"
        case logging_.CRITICAL:
            level = "🛑"
    return f"{level} {record.getMessage()}"

logger = logging_.getLogger()

for handler in logger.handlers:
    logger.removeHandler(handler)

formatter = logging_.Formatter()
formatter.format = custom_format

file_handler = logging_.FileHandler("debug.log")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging_.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

logger.setLevel(logging_.DEBUG)

logger.info(f"Torch: {torch.__version__}")
logger.info("CLIP: v1.0")

### テキストトークナイザー

In [ ]:
# BPE語彙ファイルのパスを確認

@lru_cache()
def default_bpe():
    return os.path.join("clip", "clip", "bpe_simple_vocab_16e6.txt.gz")

assert os.path.exists(default_bpe()), f"BPE vocab not found at {default_bpe()}"
default_bpe()

In [ ]:
@lru_cache()
def bytes_to_unicode():
    """
    UTF-8のバイト値をUnicode文字にマッピングする辞書を返す

    制御文字（\nなど）を256以降のUnicode文字にマッピングすることで、
    BPEが制御文字をまたいでマージするのを防ぎ、トークン化の効率と一貫性を向上させる

    Returns:
        dict: バイト値をUnicode文字にマッピングする辞書
    """
    # 制御文字（\nなど）以外のバイトを取得（188個）
    bs = list(range(ord("!"), ord("~")+1)) +  \
        list(range(ord("¡"), ord("¬")+1))+ \
        list(range(ord("®"), ord("ÿ")+1))

    cs = bs[:]

    n = 0
    for b in range(2**8):
        # 制御文字（\nなど）のバイトを256以降のUnicode文字にマッピング
        if b not in bs:
            bs.append(b)
            cs.append(2**8+n)
            n += 1

    # バイト値を対応するUnicode文字に変換 
    cs = [chr(n) for n in cs]

    # 辞書を作成
    b_to_u = dict(zip(bs, cs))

    return b_to_u

# 10進数65に対応するUnicode文字を取得
bytes_to_unicode()[65]

In [ ]:
def get_pairs(word):
    """
    隣り合う文字のペアの集合を返す
    BPEで、マージルールに基づいて文字のペアをマージする際に使用する

    Args:
        word (tuple): 文字のタプル
    Returns:
        set: 隣り合う文字のペアの集合
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

get_pairs(("t", "h", "e"))

In [ ]:
def basic_clean(text):
    """
    エンコード前の文字列の正規化

    Args:
        text (str): 入力文字列
    Returns:
        str: 正規化された文字列
    """

    # 文字化けの修正
    text = ftfy.fix_text(text)

    # HTMLエンティティ（&amp;など）のデコード
    text = html.unescape(html.unescape(text))
    return text.strip()

basic_clean("<p>This is an example! &amp;</p>")

In [ ]:
def whitespace_clean(text):
    """
    余分な空白を削除

    Args:
        text (str): 入力文字列
    Returns:
        str: 余分な空白が削除された文字列
    """

    # 正規表現を使用して、連続する空白を単一のスペースに置換
    text = re.sub(r'\s+', ' ', text)

    # 文字列の先頭と末尾の空白を削除
    text = text.strip()
    return text

whitespace_clean("  This   is   an   example!   ")

In [ ]:
class SimpleTokenizer(object):
    """
    BPEトークナイザー
    テキストをトークンIDにエンコード、もしくはトークンIDをテキストにデコードする
    """

    def __init__(self, bpe_path: str = default_bpe()):
        logger.info(f"SimpleTokenizerを初期化開始 {bpe_path=}")

        # バイト値 -> Unicode文字
        self.byte_encoder = bytes_to_unicode()

        # Unicode文字 -> バイト値
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}

        # マージルールを読み込み
        merges = gzip.open(bpe_path).read().decode("utf-8").split('\n')
        logger.debug(f"読み込んだマージルール数: {len(merges)}")
        # 不要なマージルールを削除
        merges = merges[1 : 49152-256-2+1]
        merges = [tuple(merge.split()) for merge in merges]
        logger.debug(f"使用するマージルール数: {len(merges)}")

        # 語彙を構築
        vocab = list(bytes_to_unicode().values())
        # 語彙に</w>を追加（End of Wordで原著論文と同じ方法でトークン化）
        vocab = vocab + [v + '</w>' for v in vocab]
        for merge in merges:
            vocab.append(''.join(merge))

        # 語彙に特殊トークンを追加
        vocab.extend(['<|startoftext|>', '<|endoftext|>'])

        # バイト列 -> トークンID
        # 例: ars</w> -> 864
        self.encoder = dict(zip(vocab, range(len(vocab))))

        # トークンID -> バイト列
        # 例: 864 -> ars</w>
        self.decoder = {v: k for k, v in self.encoder.items()}

        # マージルール
        # 例: ("h", "y") -> 929
        self.bpe_ranks = dict(zip(merges, range(len(merges))))

        self.cache = {'<|startoftext|>': '<|startoftext|>', '<|endoftext|>': '<|endoftext|>'}

        # 事前トークン化の正規表現パターン
        self.pat = re.compile(r"""<\|startoftext\|>|<\|endoftext\|>|'s|'t|'re|'ve|'m|'ll|'d|[\p{L}]+|[\p{N}]|[^\s\p{L}\p{N}]+""", re.IGNORECASE)

        logger.info(f"SimpleTokenizerの初期化終了")

    def bpe(self, token):
        """
        マージルールに基づいてトークンIDをマージする
        """
        logger.info(f"bpe開始 {token=}")

        # キャッシュを確認
        if token in self.cache:
            return self.cache[token]

        # 文字のタプルに変換し、最後に</w>を追加
        word = tuple(token[:-1]) + ( token[-1] + '</w>',)
        pairs = get_pairs(word)

        if not pairs:
            return token+'</w>'

        # マージできなくなるまでマージを繰り返す
        while True:
            bigram = min(pairs, key = lambda pair: self.bpe_ranks.get(pair, float('inf')))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word)-1 and word[i+1] == second:
                    new_word.append(first+second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)

        word = ' '.join(word)

        # キャッシュに保存
        self.cache[token] = word

        logger.info(f"bpe終了 {word=}")
        return word

    def encode(self, text):
        logger.info(f"encode開始 {text=}")

        bpe_tokens = []

        # 事前正規化
        text = whitespace_clean(basic_clean(text)).lower()

        # 事前トークン化により一意性を高める
        pre_tokenized = re.findall(self.pat, text)
        logger.debug(f"{pre_tokenized=}...")

        for token in pre_tokenized:
            # バイト列に変換
            token = ''.join(self.byte_encoder[b] for b in token.encode('utf-8'))

            # BPEを適用し、トークンIDに変換
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(' '))

        logger.info(f"encode終了 {bpe_tokens=}")
        return bpe_tokens

    def decode(self, tokens):
        logger.info(f"decode開始 {tokens=}")

        # トークンIDをバイト列に変換
        text = ''.join([self.decoder[token] for token in tokens])

        # バイト列をUnicode文字列に変換し、特殊トークンをスペースに置換
        text = bytearray([self.byte_decoder[c] for c in text]).decode('utf-8', errors="replace").replace('</w>', ' ')

        logger.info(f"decode終了 {text=}")
        return text

_tokenizer = SimpleTokenizer()
encoded = _tokenizer.encode("こんにちは！")
decoded = _tokenizer.decode(encoded)
encoded, decoded


In [ ]:
def tokenize(texts: Union[str, List[str]], context_length: int = 77, truncate: bool = False) -> Union[torch.IntTensor, torch.LongTensor]:
    """
    入力の文字列をトークンIDに変換する

    Args:
        texts: 入力の文字列または文字列のリスト
        context_length: コンテキスト長（CLIPモデルはすべて77を使用）
        truncate: コンテキスト長を超える場合に切り捨てるフラグ

    Returns:
        トークンIDの2次元テンソル (入力文字の数, コンテキスト長)
        torchバージョンが1.8.0未満の場合はLongTensorを返す
    """
    logger.info(f"tokenize開始 {texts=}, {context_length=}, {truncate=}")
    if isinstance(texts, str):
        texts = [texts]

    # 開始トークンと終了トークンを追加してトークン化
    sot_token = _tokenizer.encoder["<|startoftext|>"]
    eot_token = _tokenizer.encoder["<|endoftext|>"]
    logger.debug(f"{sot_token=}, {eot_token=}")
    all_tokens = [[sot_token] + _tokenizer.encode(text) + [eot_token] for text in texts]

    # 出力を格納するテンソルを0で初期化
    if version.parse(torch.__version__) < version.parse("1.8.0"):
        result = torch.zeros(len(all_tokens), context_length, dtype=torch.long)
    else:
        result = torch.zeros(len(all_tokens), context_length, dtype=torch.int)

    # トークンIDのリストをテンソルに変換
    for i, tokens in enumerate(all_tokens):
        if len(tokens) > context_length:
            if truncate:
                tokens = tokens[:context_length]
                tokens[-1] = eot_token
            else:
                raise RuntimeError(f"Input {texts[i]} is too long for context length {context_length}")
        result[i, :len(tokens)] = torch.tensor(tokens)

    logger.info(f"tokenize終了 {result.shape=}")
    return result

tokenize(["こんにちは！"])

#### Bottleneck

In [ ]:
# 今回は使わない

class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1):
        logger.info(f"Bottleneckを初期化開始 {inplanes=}, {planes=}, {stride=}")
        super().__init__()

        # all conv layers have stride 1. an avgpool is performed after the second convolution when stride > 1
        self.conv1 = nn.Conv2d(inplanes, planes, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.relu2 = nn.ReLU(inplace=True)

        self.avgpool = nn.AvgPool2d(stride) if stride > 1 else nn.Identity()

        self.conv3 = nn.Conv2d(planes, planes * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu3 = nn.ReLU(inplace=True)

        self.downsample = None
        self.stride = stride

        if stride > 1 or inplanes != planes * Bottleneck.expansion:
            # downsampling layer is prepended with an avgpool, and the subsequent convolution has stride 1
            self.downsample = nn.Sequential(OrderedDict([
                ("-1", nn.AvgPool2d(stride)),
                ("0", nn.Conv2d(inplanes, planes * self.expansion, 1, stride=1, bias=False)),
                ("1", nn.BatchNorm2d(planes * self.expansion))
            ]))
        logger.info(f"Bottleneckの初期化終了")

    def forward(self, x: torch.Tensor):
        logger.info(f"Bottleneckの順伝播開始 {x.shape=}")
        identity = x

        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.relu2(self.bn2(self.conv2(out)))
        out = self.avgpool(out)
        out = self.bn3(self.conv3(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu3(out)
        logger.info(f"Bottleneckの順伝播終了 {out.shape=}")
        return out

#### AttentionPool2d

In [ ]:
# 今回は使わないc

class AttentionPool2d(nn.Module):
    def __init__(self, spacial_dim: int, embed_dim: int, num_heads: int, output_dim: int = None):
        logger.info(f"AttentionPool2dを初期化開始 {spacial_dim=}, {embed_dim=}, {num_heads=}, {output_dim=}")
        super().__init__()
        self.positional_embedding = nn.Parameter(torch.randn(spacial_dim ** 2 + 1, embed_dim) / embed_dim ** 0.5)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.c_proj = nn.Linear(embed_dim, output_dim or embed_dim)
        self.num_heads = num_heads
        logger.info(f"AttentionPool2dの初期化終了")

    def forward(self, x):
        logger.info(f"AttentionPool2dのforward開始 {x.shape=}")
        x = x.flatten(start_dim=2).permute(2, 0, 1)  # NCHW -> (HW)NC
        x = torch.cat([x.mean(dim=0, keepdim=True), x], dim=0)  # (HW+1)NC
        x = x + self.positional_embedding[:, None, :].to(x.dtype)  # (HW+1)NC
        x, _ = F.multi_head_attention_forward(
            query=x[:1], key=x, value=x,
            embed_dim_to_check=x.shape[-1],
            num_heads=self.num_heads,
            q_proj_weight=self.q_proj.weight,
            k_proj_weight=self.k_proj.weight,
            v_proj_weight=self.v_proj.weight,
            in_proj_weight=None,
            in_proj_bias=torch.cat([self.q_proj.bias, self.k_proj.bias, self.v_proj.bias]),
            bias_k=None,
            bias_v=None,
            add_zero_attn=False,
            dropout_p=0,
            out_proj_weight=self.c_proj.weight,
            out_proj_bias=self.c_proj.bias,
            use_separate_proj_weight=True,
            training=self.training,
            need_weights=False
        )
        logger.info(f"AttentionPool2dのforward終了")
        return x.squeeze(0)

#### ModifiedResNet

In [ ]:
# 今回は使わない

class ModifiedResNet(nn.Module):
    """
    A ResNet class that is similar to torchvision's but contains the following changes:
    - There are now 3 "stem" convolutions as opposed to 1, with an average pool instead of a max pool.
    - Performs anti-aliasing strided convolutions, where an avgpool is prepended to convolutions with stride > 1
    - The final pooling layer is a QKV attention instead of an average pool
    """

    def __init__(self, layers, output_dim, heads, input_resolution=224, width=64):
        logger.info(f"ModifiedResNetを初期化開始 {layers=}, {output_dim=}, {heads=}, {input_resolution=}, {width=}")
        super().__init__()
        self.output_dim = output_dim
        self.input_resolution = input_resolution

        # the 3-layer stem
        self.conv1 = nn.Conv2d(3, width // 2, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width // 2)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(width // 2, width // 2, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(width // 2)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(width // 2, width, kernel_size=3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(width)
        self.relu3 = nn.ReLU(inplace=True)
        self.avgpool = nn.AvgPool2d(2)

        # residual layers
        self._inplanes = width  # this is a *mutable* variable used during construction
        self.layer1 = self._make_layer(width, layers[0])
        self.layer2 = self._make_layer(width * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(width * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(width * 8, layers[3], stride=2)

        embed_dim = width * 32  # the ResNet feature dimension
        self.attnpool = AttentionPool2d(input_resolution // 32, embed_dim, heads, output_dim)
        logger.info(f"ModifiedResNetの初期化終了")

    def _make_layer(self, planes, blocks, stride=1):
        logger.info(f"_make_layer開始 {planes=}, {blocks=}, {stride=}")
        layers = [Bottleneck(self._inplanes, planes, stride)]

        self._inplanes = planes * Bottleneck.expansion
        for _ in range(1, blocks):
            layers.append(Bottleneck(self._inplanes, planes))

        res = nn.Sequential(*layers)
        logger.info(f"_make_layer終了")
        return res

    def forward(self, x):
        logger.info(f"ModifiedResNet順伝播開始 {x.shape=}")
        def stem(x):
            x = self.relu1(self.bn1(self.conv1(x)))
            x = self.relu2(self.bn2(self.conv2(x)))
            x = self.relu3(self.bn3(self.conv3(x)))
            x = self.avgpool(x)
            return x

        x = x.type(self.conv1.weight.dtype)
        x = stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.attnpool(x)

        logger.info(f"ModifiedResNet順伝播終了 {x.shape=}")
        return x

#### LayerNorm

In [ ]:
class LayerNorm(nn.LayerNorm):
    """
    単精度でレイヤー正規化を適用する派生クラス
    """

    def forward(self, x: torch.Tensor):
        logger.info(f"LayerNormの適用開始 {x.shape=}, {x.dtype=}")

        orig_type = x.dtype

        # アップキャストし、親クラスの順伝播を適用
        # (50, 6, 768) -> (50, 6, 768)
        # float16 -> float32
        ret = super().forward(x.type(torch.float32))
        logger.info(f"LayerNormの適用終了 {ret.shape=}, {ret.dtype=}")

        # float32 -> float16
        return ret.type(orig_type)

#### QuickGELU

In [ ]:
class QuickGELU(nn.Module):
    """
    GELU活性化関数の近似で計算を効率化
    """

    def forward(self, x: torch.Tensor):
        logger.info(f"QuickGELUの適用開始 {x.shape=}")

        # (50, 6, 3072) -> (50, 6, 3072)
        res = x * torch.sigmoid(1.702 * x)

        logger.info(f"QuickGELUの適用終了 {res.shape=}")
        return res

#### ResidualAttentionBlock

In [ ]:
class ResidualAttentionBlock(nn.Module):
    """
    レイヤー正規化層が事前に適用されるTransformerブロック
    """


    def __init__(self, d_model: int, n_head: int, attn_mask: torch.Tensor = None):
        logger.info(f"ResidualAttentionBlockを初期化開始 {d_model=}, {n_head=}, {attn_mask is None=}")
        super().__init__()

        # マルチヘッドアテンション層
        # (シーケンス長, バッチサイズ, 埋め込み次元)で入力する必要がある
        # https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
        self.attn = nn.MultiheadAttention(
            d_model, # 768
            n_head, # 12
        )

        # レイヤー正規化層
        # 768
        self.ln_1 = LayerNorm(d_model)

        # フィードフォワードネットワーク（MLP）
        self.mlp = nn.Sequential(OrderedDict([
            ("c_fc", nn.Linear(d_model, d_model * 4)), # 768 -> 3072
            ("gelu", QuickGELU()), # 活性化関数
            ("c_proj", nn.Linear(d_model * 4, d_model)) # 3072 -> 768
        ]))

        # レイヤー正規化層
        # 768
        self.ln_2 = LayerNorm(d_model)

        # アテンションマスク（テキストエンコーダーでのみ使用）
        # (77, 77)
        self.attn_mask = attn_mask
        logger.info(f"attn_maskの形状: {self.attn_mask.shape if self.attn_mask is not None else None}")

        logger.info(f"ResidualAttentionBlockの初期化終了")

    def attention(self, x: torch.Tensor):
        logger.info(f"attention計算開始 {x.shape=}")
        self.attn_mask = self.attn_mask.to(dtype=x.dtype, device=x.device) if self.attn_mask is not None else None
        logger.debug(f"attn_maskの形状: {self.attn_mask.shape if self.attn_mask is not None else None}")

        # マルチヘッドアテンションを計算
        # 画像: (50, 6, 768) -> (50, 6, 768), attn_mask: None
        # テキスト: (77, 6, 512) -> (77, 6, 512), attn_mask: (77, 77)
        result = self.attn(x, x, x, need_weights=False, attn_mask=self.attn_mask)[0]

        logger.info(f"attention計算終了 {result.shape=}")
        return result

    def forward(self, x: torch.Tensor):
        logger.info(f"ResidualAttentionBlockの順伝播開始 {x.shape=}")

        # Transformerの前半
        # 画像: (50, 6, 768) -> (50, 6, 768)
        # テキスト: (77, 6, 512) -> (77, 6, 512)
        # レイヤー正規化を適用し、アテンションを計算し、残差接続を適用
        x = x + self.attention(self.ln_1(x))

        # Transformerの後半
        # 画像: (50, 6, 768) -> (50, 6, 768)
        # テキスト: (77, 6, 512) -> (77, 6, 512)
        # レイヤー正規化を適用し、MLPを計算し、残差接続を適用
        x = x + self.mlp(self.ln_2(x))

        logger.info(f"ResidualAttentionBlockの順伝播終了 {x.shape=}")
        return x

#### Transformer

In [ ]:
class Transformer(nn.Module):
    """
    12層のTransformerで構成されるラッパーモジュール
    """

    def __init__(self, width: int, layers: int, heads: int, attn_mask: torch.Tensor = None):
        logger.info(f"Transformerを初期化開始 {width=}, {layers=}, {heads=}, {attn_mask is None=}")
        super().__init__()

        # 768
        self.width = width

        # 12
        self.layers = layers

        # 12
        self.resblocks = nn.Sequential(
            *[ResidualAttentionBlock(width, heads, attn_mask) for _ in range(layers)]
        )
        logger.info(f"Transformerの初期化終了")

    def forward(self, x: torch.Tensor):
        logger.info(f"Transformerの順伝播開始 {x.shape=}")

        # 12層の残差アテンションブロックを順伝播
        # (50, 6, 768) -> (50, 6, 768)
        x = self.resblocks(x)

        logger.info(f"Transformerの順伝播終了 {x.shape=}")
        return x

#### VisionTransformer

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, input_resolution: int, patch_size: int, width: int, layers: int, heads: int, output_dim: int):
        logger.info(f"VisionTransformerを初期化開始 {input_resolution=}, {patch_size=}, {width=}, {layers=}, {heads=}, {output_dim=}")
        super().__init__()

        # 224
        self.input_resolution = input_resolution

        # 512
        self.output_dim = output_dim

        # 画像をパッチに分割し、特徴マップに変換する入力の畳み込み層
        # 224x224の画像を32x32のパッチに分割し、3チャネルから768チャネルに変換
        # (B, 3, 224, 224) -> (B, 768, 7, 7)
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=width, # 768
            kernel_size=patch_size, # 32
            stride=patch_size, # 32
            bias=False
        )

        # 768 ** -0.5 = 0.036084912
        scale = width ** -0.5
        logger.debug(f"{scale=}")

        # クラス埋め込みベクトル
        # (768,)
        self.class_embedding = nn.Parameter(scale * torch.randn(width))

        # 位置埋め込みベクトル
        # ((224 // 32) ** 2 + 1, 768) = (50, 768)
        self.positional_embedding = nn.Parameter(
            scale * torch.randn((input_resolution // patch_size) ** 2 + 1, width)
        )

        # 768
        self.ln_pre = LayerNorm(width)

        self.transformer = Transformer(width, layers, heads)

        # 768
        self.ln_post = LayerNorm(width)

        # 768 -> 512
        self.proj = nn.Parameter(scale * torch.randn(width, output_dim))

        logger.info(f"VisionTransformerの初期化終了")

    def forward(self, x: torch.Tensor):
        logger.info(f"VisionTransformerの順伝播開始 {x.shape=}")
        # (バッチ, チャネル, 高さ, 幅)

        # 画像をパッチに分割し、768次元の特徴マップに変換
        # (6, 3, 224, 224) -> (6, 768, 7, 7)
        x = self.conv1(x)
        logger.debug(f"conv1 {x.shape=}")

        # 14x14の特徴マップを49にフラット化
        # (6, 768, 7, 7) -> (6, 768, 49)
        x = x.reshape(x.shape[0], x.shape[1], -1)
        logger.debug(f"reshape {x.shape=}")

        # 49個のパッチをシーケンスの長さとして扱うために次元を入れ替え
        # (6, 768, 49) -> (6, 49, 768)
        x = x.permute(0, 2, 1)
        logger.debug(f"permute {x.shape=}")

        # クラス埋め込みベクトルをシーケンスの先頭に追加
        # (6, 49, 768) -> (6, 50, 768)
        x = torch.cat([self.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device), x], dim=1)
        logger.debug(f"cat {x.shape=}")

        # 位置埋め込みベクトルを加算
        # (6, 50, 768) + (50, 768) -> (6, 50, 768)
        x = x + self.positional_embedding.to(x.dtype)

        # レイヤー正規化を適用
        # (6, 50, 768) -> (6, 50, 768)
        x = self.ln_pre(x)

        # Transformerに入力するために次元を入れ替え
        # (6, 50, 768) -> (50, 6, 768)
        x = x.permute(1, 0, 2)  # NLD -> LND
        logger.debug(f"permute {x.shape=}")

        # Transformerを適用
        # (50, 6, 768) -> (50, 6, 768)
        x = self.transformer(x)

        # Transformerの出力を元の次元順に戻す
        # (50, 6, 768) -> (6, 50, 768)
        x = x.permute(1, 0, 2)  # LND -> NLD
        logger.debug(f"permute {x.shape=}")

        # レイヤー正規化を適用
        # (6, 50, 768) -> (6, 768)
        x = self.ln_post(x[:, 0, :])
        logger.debug(f"expand {x.shape=}")

        # True
        if self.proj is not None:
            # 線形投影を適用
            # (6, 768) @ (768, 512) -> (6, 512)
            x = x @ self.proj
            logger.debug(f"proj {x.shape=}")

        logger.info(f"VisionTransformerの順伝播終了 {x.shape=}")
        return x

#### CLIP

In [ ]:
class CLIP(nn.Module):
    def __init__(self,
                 embed_dim: int,
                 # vision
                 image_resolution: int,
                 vision_layers: Union[Tuple[int, int, int, int], int],
                 vision_width: int,
                 vision_patch_size: int,
                 # text
                 context_length: int,
                 vocab_size: int,
                 transformer_width: int,
                 transformer_heads: int,
                 transformer_layers: int
    ):
        logger.info(f"CLIPを初期化開始 {embed_dim=}, {image_resolution=}, {vision_layers=}, {vision_width=}, {vision_patch_size=}, {context_length=}, {vocab_size=}, {transformer_width=}, {transformer_heads=}, {transformer_layers=}")

        super().__init__()

        # 77
        self.context_length = context_length

        # False
        if isinstance(vision_layers, (tuple, list)):
            logger.debug(f"Modified ResNetを使用")

            vision_heads = vision_width * 32 // 64
            logger.debug(f"{vision_heads=}")

            self.visual = ModifiedResNet(
                layers=vision_layers,
                output_dim=embed_dim,
                heads=vision_heads,
                input_resolution=image_resolution,
                width=vision_width
            )
        else:
            logger.debug(f"Vision Transformerを使用")

            # 768 // 64 = 12
            vision_heads = vision_width // 64
            logger.debug(f"{vision_heads=}")

            self.visual = VisionTransformer(
                input_resolution=image_resolution, # 224
                patch_size=vision_patch_size, # 12
                width=vision_width, # 768
                layers=vision_layers, # 12
                heads=vision_heads, # 12
                output_dim=embed_dim # 512
            )

        self.transformer = Transformer(
            width=transformer_width, # 512
            layers=transformer_layers, # 12
            heads=transformer_heads, # 8
            attn_mask=self.build_attention_mask()
        )

        # 語彙サイズ
        # 49408
        self.vocab_size = vocab_size

        # トークンの埋め込み
        # 49408 -> 512
        self.token_embedding = nn.Embedding(vocab_size, transformer_width)

        # 位置埋め込み
        # (77, 512)
        self.positional_embedding = nn.Parameter(torch.empty(self.context_length, transformer_width))

        # レイヤー正規化層
        # 512
        self.ln_final = LayerNorm(transformer_width)

        # テキストの射影行列
        # 512 -> 512
        self.text_projection = nn.Parameter(torch.empty(transformer_width, embed_dim))

        # 2.6593で初期化
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        logger.debug(f"{self.logit_scale=}")

        # パラメータの初期化
        self.initialize_parameters()

        logger.info(f"CLIPの初期化終了")

    def initialize_parameters(self):
        logger.info(f"CLIPのパラメータ初期化開始")

        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.positional_embedding, std=0.01)

        # False
        if isinstance(self.visual, ModifiedResNet):
            if self.visual.attnpool is not None:
                std = self.visual.attnpool.c_proj.in_features ** -0.5
                logger.debug(f"{std=}")

                nn.init.normal_(self.visual.attnpool.q_proj.weight, std=std)
                nn.init.normal_(self.visual.attnpool.k_proj.weight, std=std)
                nn.init.normal_(self.visual.attnpool.v_proj.weight, std=std)
                nn.init.normal_(self.visual.attnpool.c_proj.weight, std=std)

            for resnet_block in [self.visual.layer1, self.visual.layer2, self.visual.layer3, self.visual.layer4]:
                for name, param in resnet_block.named_parameters():
                    if name.endswith("bn3.weight"):
                        nn.init.zeros_(param)

        # 0.009021
        proj_std = (self.transformer.width ** -0.5) * ((2 * self.transformer.layers) ** -0.5)
        logger.debug(f"{proj_std=}")

        # 0.04419
        attn_std = self.transformer.width ** -0.5
        logger.debug(f"{attn_std=}")

        # 0.03125
        fc_std = (2 * self.transformer.width) ** -0.5
        logger.debug(f"{fc_std=}")

        for block in self.transformer.resblocks:
            nn.init.normal_(block.attn.in_proj_weight, std=attn_std)
            nn.init.normal_(block.attn.out_proj.weight, std=proj_std)
            nn.init.normal_(block.mlp.c_fc.weight, std=fc_std)
            nn.init.normal_(block.mlp.c_proj.weight, std=proj_std)

        if self.text_projection is not None:
            nn.init.normal_(self.text_projection, std=self.transformer.width ** -0.5)

        logger.info(f"CLIPのパラメータ初期化終了")

    def build_attention_mask(self):
        logger.info(f"CLIPのアテンションマスク作成開始")
        # lazily create causal attention mask, with full attention between the vision tokens
        # pytorch uses additive attention mask; fill with -inf

        # (77, 77)
        mask = torch.empty(self.context_length, self.context_length)

        # 上三角行列を1で埋め、下三角行列をマイナス無限大で埋める
        mask.fill_(float("-inf"))
        mask.triu_(1)

        logger.info(f"CLIPのアテンションマスク作成終了 {mask.shape=}")
        return mask

    @property
    def dtype(self):
        return self.visual.conv1.weight.dtype

    def encode_image(self, image):
        logger.info(f"画像エンコード開始 {image.shape=}")

        # Vision Transformerを適用
        # (6, 3, 224, 224) -> (6, 512)
        image = self.visual(image.type(self.dtype))
        logger.info(f"画像エンコード終了 {image.shape=}")
        return image

    def encode_text(self, text):
        logger.info(f"テキストエンコード開始 {text.shape=}")

        # 入力のテキストを埋め込みベクトルに変換
        # (6, 77) -> (6, 77, 512)
        x = self.token_embedding(text).type(self.dtype)
        logger.debug(f"token_embedding {x.shape=}")

        # 位置埋め込みベクトルを加算
        # (6, 77, 512) + (77, 512) -> (6, 77, 512)
        x = x + self.positional_embedding.type(self.dtype)

        # 次元を入れ替え
        # (6, 77, 512) -> (77, 6, 512)
        x = x.permute(1, 0, 2)  # NLD -> LND
        logger.debug(f"permute {x.shape=}")

        # トランスフォーマーを適用
        x = self.transformer(x)

        # 次元を元に戻す
        # (77, 6, 512) -> (6, 77, 512)
        x = x.permute(1, 0, 2)  # LND -> NLD
        logger.debug(f"permute {x.shape=}")

        # レイヤー正規化の適用
        # (6, 77, 512) -> (6, 77, 512)
        x = self.ln_final(x).type(self.dtype)

        # End of Textトークンのみを残して、テキストの射影
        # text.argmax(dim=-1)はテキストの<EOT>トークンの位置
        # (6, 77, 512) -> (6, 512)
        # (6, 512) @ (512, 512) -> (6, 512)
        x = x[torch.arange(x.shape[0]), text.argmax(dim=-1)] @ self.text_projection
        logger.debug(f"text_projection {x.shape=}")

        logger.info(f"テキストエンコード終了 {x.shape=}")
        return x

    def forward(self, image, text):
        """
        訓練時の順伝播
        """
        logger.info(f"CLIPの順伝播開始 {image.shape=}, {text.shape=}")

        image_features = self.encode_image(image)
        logger.debug(f"{image_features.shape=}")

        text_features = self.encode_text(text)
        logger.debug(f"{text_features.shape=}")

        # normalized features
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        text_features = text_features / text_features.norm(dim=1, keepdim=True)

        # cosine similarity as logits
        logit_scale = self.logit_scale.exp()
        logger.debug(f"{logit_scale=}")

        logits_per_image = logit_scale * image_features @ text_features.t()
        logger.debug(f"{logits_per_image.shape=}")

        logits_per_text = logits_per_image.t()
        logger.debug(f"{logits_per_text.shape=}")

        # shape = [global_batch_size, global_batch_size]
        logger.info(f"CLIPの順伝播終了 {logits_per_image.shape=}, {logits_per_text.shape=}")

        # このロジットがクロスエントロピー損失に渡される
        return logits_per_image, logits_per_text

## テスト

### 学習済みモデルの読み込み

In [ ]:
_MODELS = {
    "RN50": "https://openaipublic.azureedge.net/clip/models/afeb0e10f9e5a86da6080e35cf09123aca3b358a0c3e3b6c78a7b63bc04b6762/RN50.pt",
    "RN101": "https://openaipublic.azureedge.net/clip/models/8fa8567bab74a42d41c5915025a8e4538c3bdbe8804a470a72f30b0d94fab599/RN101.pt",
    "RN50x4": "https://openaipublic.azureedge.net/clip/models/7e526bd135e493cef0776de27d5f42653e6b4c8bf9e0f653bb11773263205fdd/RN50x4.pt",
    "RN50x16": "https://openaipublic.azureedge.net/clip/models/52378b407f34354e150460fe41077663dd5b39c54cd0bfd2b27167a4a06ec9aa/RN50x16.pt",
    "RN50x64": "https://openaipublic.azureedge.net/clip/models/be1cfb55d75a9666199fb2206c106743da0f6468c9d327f3e0d0a543a9919d9c/RN50x64.pt",
    "ViT-B/32": "https://openaipublic.azureedge.net/clip/models/40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af/ViT-B-32.pt",
    "ViT-B/16": "https://openaipublic.azureedge.net/clip/models/5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f/ViT-B-16.pt",
    "ViT-L/14": "https://openaipublic.azureedge.net/clip/models/b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt",
    "ViT-L/14@336px": "https://openaipublic.azureedge.net/clip/models/3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02/ViT-L-14-336px.pt",
}

MODEL = "ViT-B/32"
device = "cuda"

In [ ]:
def _download(url: str, root: str):
    os.makedirs(root, exist_ok=True)
    filename = os.path.basename(url)

    # URLからSHA256ハッシュを取得
    expected_sha256 = url.split("/")[-2]

    download_target = os.path.join(root, filename)

    if os.path.exists(download_target) and not os.path.isfile(download_target):
        raise RuntimeError(f"{download_target} exists and is not a regular file")

    if os.path.isfile(download_target):
        if hashlib.sha256(open(download_target, "rb").read()).hexdigest() == expected_sha256:
            return download_target
        else:
            warnings.warn(f"{download_target} exists, but the SHA256 checksum does not match; re-downloading the file")

    # urllibを使ってファイルをダウンロード
    with urllib.request.urlopen(url) as source, open(download_target, "wb") as output:
        with tqdm(total=int(source.info().get("Content-Length")), ncols=80, unit='iB', unit_scale=True, unit_divisor=1024) as loop:
            while True:
                buffer = source.read(8192)
                if not buffer:
                    break

                output.write(buffer)
                loop.update(len(buffer))

    # hashlibを使ってSHA256ハッシュを検証
    if hashlib.sha256(open(download_target, "rb").read()).hexdigest() != expected_sha256:
        raise RuntimeError("Model has been downloaded but the SHA256 checksum does not not match")

    return download_target

model_path = _download(_MODELS[MODEL], os.path.expanduser("~/.cache/clip"))
model_path

In [ ]:
def convert_weights(model: nn.Module):
    """
    モデルのパラメータを半精度(float16)に直接変換し、メモリ使用量を削減する

    Parameters:
        model (nn.Module): 変換するPyTorchモデル
    """

    def _convert_weights_to_fp16(l):
        """
        モデルの特定の層の重みとバイアスをfloat16に変換するヘルパー関数

        Args:
            l (nn.Module): 変換する層
        """

        # 畳み込み層と線形層の重みとバイアスをfloat16に変換
        if isinstance(l, (nn.Conv1d, nn.Conv2d, nn.Linear)):
            l.weight.data = l.weight.data.half()
            if l.bias is not None:
                l.bias.data = l.bias.data.half()

        # マルチヘッドアテンション層の特定のパラメータをfloat16に変換
        if isinstance(l, nn.MultiheadAttention):
            for attr in [*[f"{s}_proj_weight" for s in ["in", "q", "k", "v"]], "in_proj_bias", "bias_k", "bias_v"]:
                tensor = getattr(l, attr)
                if tensor is not None:
                    tensor.data = tensor.data.half()

        # テキストおよび画像の射影行列をfloat16に変換
        for name in ["text_projection", "proj"]:
            if hasattr(l, name):
                attr = getattr(l, name)
                if attr is not None:
                    attr.data = attr.data.half()

    # モデル全体に対してヘルパー関数を適用
    model.apply(_convert_weights_to_fp16)

In [ ]:
def build_model(state_dict: dict):
    """
    state_dictからCLIPモデルを構築する

    Args:
        state_dict (dict): モデルの重みを含むstate_dict
    Returns:
        CLIP: 事前学習済みをロードしたCLIPモデル
    """

    # 1. 初期化パラメータを取得

    # True
    vit = "visual.proj" in state_dict

    if vit:
        # 768
        vision_width = state_dict["visual.conv1.weight"].shape[0]

        # 12
        vision_layers = len([k for k in state_dict.keys() if k.startswith("visual.") and k.endswith(".attn.in_proj_weight")])

        # 32
        vision_patch_size = state_dict["visual.conv1.weight"].shape[-1]

        # 7
        grid_size = round((state_dict["visual.positional_embedding"].shape[0] - 1) ** 0.5)

        # 224
        image_resolution = vision_patch_size * grid_size
    else:
        counts: list = [len(set(k.split(".")[2] for k in state_dict if k.startswith(f"visual.layer{b}"))) for b in [1, 2, 3, 4]]
        vision_layers = tuple(counts)
        vision_width = state_dict["visual.layer1.0.conv1.weight"].shape[0]
        output_width = round((state_dict["visual.attnpool.positional_embedding"].shape[0] - 1) ** 0.5)
        vision_patch_size = None
        assert output_width ** 2 + 1 == state_dict["visual.attnpool.positional_embedding"].shape[0]
        image_resolution = output_width * 32

    # 512
    embed_dim = state_dict["text_projection"].shape[1]

    # 77
    context_length = state_dict["positional_embedding"].shape[0]

    # 49408
    vocab_size = state_dict["token_embedding.weight"].shape[0]

    # 512
    transformer_width = state_dict["ln_final.weight"].shape[0]

    # 8
    transformer_heads = transformer_width // 64

    # 12
    transformer_layers = len(set(k.split(".")[2] for k in state_dict if k.startswith("transformer.resblocks")))

    # 2. モデルを初期化
    model = CLIP(
        embed_dim,
        image_resolution,
        vision_layers,
        vision_width,
        vision_patch_size,
        context_length,
        vocab_size,
        transformer_width,
        transformer_heads,
        transformer_layers
    )

    # 3. 不要なキーを削除

    for key in ["input_resolution", "context_length", "vocab_size"]:
        if key in state_dict:
            del state_dict[key]

    # 4. 半精度に変換
    convert_weights(model)

    # 5. モデルの重みを読み込む
    model.load_state_dict(state_dict)

    # 6. 評価モードに設定
    return model.eval()

In [ ]:
# ダウンロードしたモデルを読み込む

with open(model_path, 'rb') as opened_file:
    model = torch.jit.load(opened_file, map_location=device).eval()
    state_dict = model.state_dict()

model = build_model(state_dict or model.state_dict()).to(device)
print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", model.visual.input_resolution)
print("Context length:", model.context_length)
print("Vocab size:", model.vocab_size)

### 画像プリプロセッサー

In [ ]:
def _convert_image_to_rgb(image):
    return image.convert("RGB")

In [ ]:
def _transform(n_px):
    return Compose([
        Resize(n_px, interpolation=BICUBIC),
        CenterCrop(n_px),
        _convert_image_to_rgb,
        ToTensor(),
        Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
    ])

preprocess = _transform(model.visual.input_resolution)
preprocess

### テキストと画像の類似度計算

In [ ]:
# テスト用の画像とプロンプトの準備、可視化

# scikit-imageの画像の名前とその説明文
# 画像の名前は画像の取得に使用
# 説明文はテキストエンコードに使用
descriptions = {
    "page": "a page of text about segmentation",
    "chelsea": "a facial photo of a tabby cat",
    "astronaut": "a portrait of an astronaut with the American flag",
    "rocket": "a rocket standing on a launchpad",
    "camera": "a person looking at a camera on a tripod",
    "coffee": "a cup of coffee on a saucer"
}

# 画像の枚数
n_items = len(descriptions)

original_images = []
images = []
texts = []
plt.figure(figsize=(4 * 2, 4 * n_items))

for name, desc in descriptions.items():

    # scikit-imageで画像を取得
    caller = getattr(skimage.data, name)
    image = caller()
    image = Image.fromarray(image)
    original_images.append(image)

    # 画像を表示
    plt.subplot(8, 2, len(original_images) * 2 + 1)
    plt.imshow(image, cmap=plt.cm.gray)
    plt.axis('off')
    plt.title(f"{name}\n{desc}")

    # 画像を前処理
    image = preprocess(image)
    images.append(image)

    plt.subplot(8, 2, len(original_images) * 2 + 2)
    image_display = image.permute(1, 2, 0)
    image_display = image_display.detach().cpu()
    image_display = (image_display - image_display.min()) / (image_display.max() - image_display.min())
    plt.imshow(image_display, cmap=plt.cm.gray)
    plt.axis('off')
    plt.title(f"preprocessed")

    # 対応するプロンプトを追加
    texts.append(desc)

# 画像をバッチ化してGPUに転送 (6, 3, 224, 224)
image_input = torch.stack(images).cuda()

In [ ]:
# テキストのトークン化

text_input = ["This is " + desc for desc in texts]
text_tokens = tokenize(text_input).cuda()
text_tokens.shape, text_tokens[0]

# 77トークンにパディングされる

In [ ]:
# テキストと画像の特徴量抽出

with torch.no_grad():
    # 画像をエンコード (6, 3, 224, 224) -> (6, 512)
    image_features = model.encode_image(image_input).float()

    # テキストをエンコード (6, 77) -> (6, 512)
    text_features = model.encode_text(text_tokens).float()

image_features.shape, text_features.shape

In [ ]:
# L2正規化を適用

# ノルムを計算
image_norm = image_features.norm(dim=-1, keepdim=True)
text_norm = text_features.norm(dim=-1, keepdim=True)

# L2正規化
normalized_image_features = image_features / image_norm
normalized_text_features = text_features / text_norm

normalized_image_features.shape, normalized_text_features.shape

In [ ]:
# コサイン類似度を計算

# (6, 512) @ (512, 6) -> (6, 6)
similarity = normalized_image_features @ normalized_text_features.t()
similarity.shape

In [ ]:
# コサイン類似度を可視化

plt.figure(figsize=(20, 14))
plt.imshow(similarity.cpu(), vmin=0.1, vmax=0.3)
plt.yticks(range(n_items), texts, fontsize=18)
plt.xticks([])

for i, image in enumerate(original_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower", cmap=plt.cm.gray)
for x in range(similarity.shape[1]):
    for y in range(similarity.shape[0]):
        plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

for side in ["left", "top", "right", "bottom"]:
  plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, n_items - 0.5])
plt.ylim([n_items + 0.5, -2])

plt.title("Cosine similarity between text and image features", size=20)
plt.show()

### ゼロショット分類

CIFAR100のラベルをクエリにして、画像検索を行う

In [ ]:
logger.setLevel(logging_.WARNING)  # ログレベルをWARNINGに戻す

In [ ]:
# CIFAR100（シファー・ワンハンドレット）のラベルを用意

cifar100_classes = ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm']

len(cifar100_classes)

In [ ]:
# プロンプトテンプレートを作成

prompt_template = ['a bad photo of a {}.', 'a photo of many {}.', 'a sculpture of a {}.', 'a photo of the hard to see {}.', 'a low resolution photo of the {}.', 'a rendering of a {}.', 'graffiti of a {}.', 'a bad photo of the {}.', 'a cropped photo of the {}.', 'a tattoo of a {}.', 'the embroidered {}.', 'a photo of a hard to see {}.', 'a bright photo of a {}.', 'a photo of a clean {}.', 'a photo of a dirty {}.', 'a dark photo of the {}.', 'a drawing of a {}.', 'a photo of my {}.', 'the plastic {}.', 'a photo of the cool {}.', 'a close-up photo of a {}.', 'a black and white photo of the {}.', 'a painting of the {}.', 'a painting of a {}.', 'a pixelated photo of the {}.', 'a sculpture of the {}.', 'a bright photo of the {}.', 'a cropped photo of a {}.', 'a plastic {}.', 'a photo of the dirty {}.', 'a jpeg corrupted photo of a {}.', 'a blurry photo of the {}.', 'a photo of the {}.', 'a good photo of the {}.', 'a rendering of the {}.', 'a {} in a video game.', 'a photo of one {}.', 'a doodle of a {}.', 'a close-up photo of the {}.', 'a photo of a {}.', 'the origami {}.', 'the {} in a video game.', 'a sketch of a {}.', 'a doodle of the {}.', 'a origami {}.', 'a low resolution photo of a {}.', 'the toy {}.', 'a rendition of the {}.', 'a photo of the clean {}.', 'a photo of a large {}.', 'a rendition of a {}.', 'a photo of a nice {}.', 'a photo of a weird {}.', 'a blurry photo of a {}.', 'a cartoon {}.', 'art of a {}.', 'a sketch of the {}.', 'a embroidered {}.', 'a pixelated photo of a {}.', 'itap of the {}.', 'a jpeg corrupted photo of the {}.', 'a good photo of a {}.', 'a plushie {}.', 'a photo of the nice {}.', 'a photo of the small {}.', 'a photo of the weird {}.', 'the cartoon {}.', 'art of the {}.', 'a drawing of the {}.', 'a photo of the large {}.', 'a black and white photo of a {}.', 'the plushie {}.', 'a dark photo of a {}.', 'itap of a {}.', 'graffiti of the {}.', 'a toy {}.', 'itap of my {}.', 'a photo of a cool {}.', 'a photo of a small {}.', 'a tattoo of the {}.']

len(prompt_template)

In [ ]:
# アンサンブルによるゼロショット分類

zeroshot_weights = []

# 100クラスについてループ
for c in cifar100_classes:

    # 1クラスに付き80のプロンプトを作成
    texts = [template.format(c) for template in prompt_template]

    # プロンプトをトークン化 (80, 77)
    text_tokens = tokenize(texts).cuda()

    with torch.no_grad():
        # プロンプトをエンコード (80, 512)
        class_embeddings = model.encode_text(text_tokens).float()

        # プロンプトごとにL2正規化 (80, 512)
        class_embeddings /= class_embeddings.norm(dim=-1, keepdim=True)

        # 80の特徴量を1つの特徴量に平均化 (512,)
        class_embedding = class_embeddings.mean(dim=0)

        # もう一度L2正規化 (512,)
        class_embedding /= class_embedding.norm()

    zeroshot_weights.append(class_embedding)

# リストをテンソルに変換し、GPUに転送 (512, 100)
zeroshot_weights = torch.stack(zeroshot_weights, dim=1).cuda() 

# コサイン類似度を計算し、ソフトマックスを適用して確率に変換
text_probs = (100.0 * image_features @ zeroshot_weights).softmax(dim=-1)

# 上位5クラスを取得
top_probs, top_labels = text_probs.cpu().topk(5, dim=-1)

In [ ]:
# 可視化

plt.figure(figsize=(16, 16))

for i, image in enumerate(original_images):
    plt.subplot(4, 4, 2 * i + 1)
    plt.imshow(image, cmap=plt.cm.gray)
    plt.axis("off")

    plt.subplot(4, 4, 2 * i + 2)
    y = np.arange(top_probs.shape[-1])
    plt.grid()
    plt.barh(y, top_probs[i])
    plt.gca().invert_yaxis()
    plt.gca().set_axisbelow(True)
    plt.yticks(y, [f"{cifar100_classes[index]} {top_probs[i][j]:.2f}" for j, index in enumerate(top_labels[i].numpy())])
    plt.xlabel("probability")

plt.subplots_adjust(wspace=0.5)
plt.show()